# Employee Attrition Prediction — 10,000 Employees

**Google Colab | University Machine Learning Project**

**Dataset:** `Employee_Attrition_10000_Training_Dataset.xlsx`  
**Target:** `Attrition` (Yes = 1, employee leaves; No = 0, employee stays)  
**Algorithms:** Logistic Regression, Random Forest, Support Vector Machine (SVM)

**How to run:** Select **Runtime → Run all**, or run each cell in order with `Shift + Enter`. Upload the Excel file when prompted.

> **Important:** This is a 10,000-row sample from a public HR Analytics educational dataset. It does not contain verified real employee records. It has 10 columns and is different from the older IBM dataset with 1,470 rows and 35 columns. Do not compare their accuracy scores directly.

## 1. Required Libraries
Run this cell first in Google Colab.

In [ ]:
%pip -q install pandas numpy scikit-learn openpyxl matplotlib seaborn joblib

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, roc_curve,
)

RANDOM_STATE = 42
print('Libraries imported successfully!')

## 2. Excel Dataset Upload
Use the upload button to select **`Employee_Attrition_10000_Training_Dataset.xlsx`**. The notebook reads the `Attrition_Data` sheet.

In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise ValueError('Please upload the 10,000-row Excel file and run this cell again.')

file_name = next((name for name in uploaded if name.lower().endswith('.xlsx')), None)
if file_name is None:
    raise ValueError('Please upload a .xlsx Excel file.')

df = pd.read_excel(file_name, sheet_name='Attrition_Data')
df.columns = df.columns.str.strip()
print('Uploaded:', file_name)
print('Dataset shape (rows, columns):', df.shape)
display(df.head())

## 3. Dataset Checking
This step checks the row count, columns, class balance, missing values, and duplicate rows.

In [ ]:
expected_cols = [
    'satisfaction_level', 'last_evaluation', 'number_project',
    'average_montly_hours', 'time_spend_company', 'Work_accident',
    'promotion_last_5years', 'Department', 'salary', 'Attrition',
]
missing_cols = [col for col in expected_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f'Expected dataset columns are missing: {missing_cols}')

print('Number of rows:', len(df))
print('Number of columns:', len(df.columns))
print('Total missing cells:', int(df[expected_cols].isna().sum().sum()))
print('Duplicate rows:', int(df[expected_cols].duplicated().sum()))
print('\nOriginal Attrition labels:')
print(df['Attrition'].value_counts(dropna=False))

if len(df) != 10000:
    print('WARNING: Expected 10,000 records; check your uploaded file.')

## 4. Label Encoding + Train / Validation / Test Split
`Attrition: Yes → 1`, `No → 0`. The data split is **64% training, 16% validation, and 20% final testing**.  

**Data leakage prevention:** The imputer, scaler, and one-hot encoder are fitted only on the training data.

In [ ]:
# Use only known columns; don't accidentally include identifiers or target in X.
df = df[expected_cols].copy()
df['Attrition'] = (
    df['Attrition'].astype('string').str.strip().str.lower()
    .map({'yes': 1, 'no': 0})
)
if df['Attrition'].isna().any():
    raise ValueError('Attrition contains unexpected labels; expected Yes or No.')
df['Attrition'] = df['Attrition'].astype(int)

numeric_cols = [
    'satisfaction_level', 'last_evaluation', 'number_project',
    'average_montly_hours', 'time_spend_company', 'Work_accident',
    'promotion_last_5years',
]
categorical_cols = ['Department', 'salary']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

X = df[numeric_cols + categorical_cols].copy()
y = df['Attrition'].copy()

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.20, stratify=y_dev,
    random_state=RANDOM_STATE,
)

preprocessor = ColumnTransformer([
    ('numeric', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), numeric_cols),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]), categorical_cols),
])

print('Training rows:', len(X_train))
print('Validation rows:', len(X_val))
print('Final test rows:', len(X_test))
print('Total:', len(X_train) + len(X_val) + len(X_test))
print('\nTraining target distribution:')
print(y_train.value_counts())

## 5. Train Machine Learning Models
Train three classifiers. `class_weight` gives more weight to the smaller attrition class. This run does not use SMOTE.

> Training the SVM with `probability=True` may take longer.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, class_weight='balanced_subsample',
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    'SVM': SVC(
        kernel='rbf', C=1.0, class_weight='balanced',
        probability=True, random_state=RANDOM_STATE
    ),
}

validation_results = []
validation_models = {}

for model_name, estimator in models.items():
    print(f'Training: {model_name} ...')
    pipeline = Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('classifier', clone(estimator)),
    ])
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_val)
    prob = pipeline.predict_proba(X_val)[:, 1]

    validation_models[model_name] = pipeline
    validation_results.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_val, pred),
        'Precision': precision_score(y_val, pred, zero_division=0),
        'Recall': recall_score(y_val, pred, zero_division=0),
        'F1-Score': f1_score(y_val, pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_val, prob),
    })
    print(f'Completed: {model_name}\n')

results_df = pd.DataFrame(validation_results).sort_values(
    ['F1-Score', 'ROC-AUC'], ascending=False
).reset_index(drop=True)
print('VALIDATION MODEL COMPARISON')
display(results_df.round(4))

## 6. Validation Model Comparison Graph
Higher **recall** means the model finds more employees who left. **F1-score** balances precision and recall.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
results_df.set_index('Model')[['F1-Score', 'Recall', 'Precision']].plot(
    kind='bar', ax=ax, rot=0
)
ax.set_title('Model Comparison (Validation Set)')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## 7. Select the Best Model and Run the Final Test
Select the model with the highest **validation F1-score**. Retrain it on the full development set (80% of the data). Then evaluate it once on the untouched 20% test set.

In [ ]:
best_name = results_df.loc[0, 'Model']
print('Selected model (highest validation F1):', best_name)

best_model = Pipeline([
    ('preprocessor', clone(preprocessor)),
    ('classifier', clone(models[best_name])),
])

best_model.fit(X_dev, y_dev)
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

final_metrics = pd.DataFrame([{
    'Best Model': best_name,
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred, zero_division=0),
    'Recall': recall_score(y_test, y_pred, zero_division=0),
    'F1-Score': f1_score(y_test, y_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, y_prob),
}])
print('FINAL TEST METRICS')
display(final_metrics.round(4))

print('\nCLASSIFICATION REPORT:')
print(classification_report(
    y_test, y_pred, target_names=['Stayed (No)', 'Left (Yes)'],
    zero_division=0
))

## 8. Confusion Matrix (Final Test)

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
plt.figure(figsize=(6, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Stayed', 'Left'],
            yticklabels=['Stayed', 'Left'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix — {best_name}')
plt.tight_layout()
plt.show()

print(f'True Negatives : {cm[0,0]}')
print(f'False Positives: {cm[0,1]}')
print(f'False Negatives: {cm[1,0]}  (actual leavers missed)')
print(f'True Positives : {cm[1,1]}')

## 9. ROC Curve (Final Test)

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_value = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'{best_name} (AUC = {auc_value:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Final Test')
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 10. Sample Employee Prediction
Use one row from the held-out test set as an example. This prediction is only a model estimate; it does not prove that an employee will leave.

In [ ]:
sample_employee = X_test.iloc[[0]].copy()
sample_pred = best_model.predict(sample_employee)[0]
sample_prob = best_model.predict_proba(sample_employee)[0, 1]

print('Sample employee inputs:')
display(sample_employee)
print('Predicted Attrition:', 'Yes (might leave)' if sample_pred == 1 else 'No (might stay)')
print(f'Model-estimated probability of leaving: {sample_prob:.2%}')

## 11. Model + Results Download
The `.pkl` file saves both the preprocessing steps and the trained classifier. The Excel file contains the validation comparison and final test results.

**Security note:** Load `.pkl` files only from trusted sources.

In [ ]:
from google.colab import files

model_file = 'Employee_Attrition_Best_Model.pkl'
report_file = 'Employee_Attrition_Results.xlsx'

joblib.dump(best_model, model_file)
with pd.ExcelWriter(report_file, engine='openpyxl') as writer:
    results_df.to_excel(writer, sheet_name='Validation_Comparison', index=False)
    final_metrics.to_excel(writer, sheet_name='Final_Test', index=False)

print('Saved:', model_file, report_file)
files.download(model_file)
files.download(report_file)

## Research Project Notes

- **Data source:** 10,000 unique observations sampled from the 14,999-row public *HR Analytics* educational dataset. Original columns were relabeled where needed (`left` → `Attrition`, `sales` → `Department`). Source: [HR Analytics CSV](https://raw.githubusercontent.com/RuchitaGarde/Human-Resources-Analytics/refs/heads/master/HR_comma_sep.csv).
- **Class imbalance:** Original sample is not 50/50. Evaluate **Recall, F1 and ROC-AUC** as well as Accuracy.
- **Avoid data leakage:** Never apply SMOTE/scaling to the full dataset before splitting. Pipeline handles preprocessing correctly.
- **Model selection:** Uses validation set F1. The final test set is untouched until the selected model has been retrained.
- **Interpretation:** Data is an educational/simulated public benchmark; results do **not** establish real-world employee attrition prediction accuracy or explain causal effects.
- **Comparison:** Old IBM 1,470-record dataset results are from a different source and feature set, so cannot be directly compared to this notebook as a controlled improvement.

**Suggested next step:** After running the notebook in Colab, use `Employee_Attrition_Results.xlsx` and the final confusion matrix to update the research article's performance table.